In [12]:
import sys, platform
#!{sys.executable} -m pip install -U pandas
#!{sys.executable} -m pip install -U xgboost
#!{sys.executable} -m pip install -U scikit-learn
import sklearn
import subprocess, importlib
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from xgboost import XGBClassifier


df = pd.read_csv("elliptic_txs_features.csv", header=None)

n_cols = df.shape[1]
cols = ["txId", "timestep"] + [f"f{i}" for i in range(n_cols - 2)]
df.columns = cols



In [13]:
cls = pd.read_csv("elliptic_txs_classes.csv")
cls = cls.rename(columns={cls.columns[0]: "txId", cls.columns[1]: "class"})
cls["class"] = cls["class"].astype(str)


df = df.merge(cls[["txId", "class"]], on="txId", how="left")

df = df[df["class"].isin(["1", "2"])].copy()

df["y"] = (df["class"] == "1").astype(int)

print("class counts:")
print(df["class"].value_counts(dropna=False).head(10))

print("y counts:", df["y"].value_counts())
print("illicit ratio:", df["y"].mean())
print("timestep min/max:", int(df["timestep"].min()), int(df["timestep"].max()))

df["y"] = df["y"].astype(int)
df = df[df["y"].isin([0, 1])].copy()

class counts:
class
2    42019
1     4545
Name: count, dtype: int64
y counts: y
0    42019
1     4545
Name: count, dtype: int64
illicit ratio: 0.09760759384932566
timestep min/max: 1 49


In [14]:
TRAIN_END_TS = 34
VAL_START_TS = 33


exclude_cols = {"txId", "class", "y"}
feature_cols = [c for c in df.columns if c not in exclude_cols]


train_mask = df["timestep"] <= TRAIN_END_TS
test_mask  = df["timestep"] >  TRAIN_END_TS

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

val_mask = df_train["timestep"] >= VAL_START_TS
df_val = df_train[val_mask].copy()
df_tr  = df_train[~val_mask].copy()

X_tr, y_tr = df_tr[feature_cols], df_tr["y"].astype(int)
X_val, y_val = df_val[feature_cols], df_val["y"].astype(int)
X_train_full, y_train_full = df_train[feature_cols], df_train["y"].astype(int)
X_test, y_test = df_test[feature_cols], df_test["y"].astype(int)

print("Features:", len(feature_cols))
print("Train main:", X_tr.shape, "val:", X_val.shape, "test:", X_test.shape)
print("Train main illicit ratio:", float(y_tr.mean()))
print("Val illicit ratio:", float(y_val.mean()))
print("Test illicit ratio:", float(y_test.mean()))

Features: 166
Train main: (28938, 166) val: (956, 166) test: (16670, 166)
Train main illicit ratio: 0.11756168359941944
Val illicit ratio: 0.06276150627615062
Test illicit ratio: 0.06496700659868027


In [15]:

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())
scale_pos_weight = n_neg / max(1, n_pos)
print("scale_pos_weight:", scale_pos_weight, "(neg:", n_neg, "pos:", n_pos, ")")

params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    objective="binary:logistic",
    tree_method="hist",
    eval_metric=["aucpr", "auc"],
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
)

model = XGBClassifier(**params)


params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    tree_method="hist",
    eval_metric=["aucpr", "auc"],
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=100,
)

model = XGBClassifier(**params)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50
)
print("best_iteration:", model.best_iteration)

scale_pos_weight: 7.506172839506172 (neg: 25536 pos: 3402 )
[0]	validation_0-aucpr:0.70724	validation_0-auc:0.97891
[50]	validation_0-aucpr:0.94758	validation_0-auc:0.99310
[100]	validation_0-aucpr:0.96264	validation_0-auc:0.99606
[150]	validation_0-aucpr:0.96737	validation_0-auc:0.99660
[200]	validation_0-aucpr:0.97125	validation_0-auc:0.99715
[250]	validation_0-aucpr:0.97286	validation_0-auc:0.99728
[300]	validation_0-aucpr:0.97348	validation_0-auc:0.99741
[350]	validation_0-aucpr:0.97449	validation_0-auc:0.99751
[400]	validation_0-aucpr:0.97372	validation_0-auc:0.99736
[450]	validation_0-aucpr:0.97251	validation_0-auc:0.99719
[458]	validation_0-aucpr:0.97262	validation_0-auc:0.99721
best_iteration: 358


In [16]:
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

val_proba  = model.predict_proba(X_val)[:, 1]
test_proba = model.predict_proba(X_test)[:, 1]

print("VAL  PR-AUC:", average_precision_score(y_val, val_proba))
print("VAL  ROC-AUC:", roc_auc_score(y_val, val_proba))
print("TEST PR-AUC:", average_precision_score(y_test, test_proba))
print("TEST ROC-AUC:", roc_auc_score(y_test, test_proba))

test_pred_05 = (test_proba >= 0.5).astype(int)
print("\n--- TEST report @ threshold=0.5 ---")
print(classification_report(y_test, test_pred_05, target_names=["licit(0)", "illicit(1)"]))

VAL  PR-AUC: 0.9759619776458118
VAL  ROC-AUC: 0.9976934523809524
TEST PR-AUC: 0.7932412987772565
TEST ROC-AUC: 0.9230256219506263

--- TEST report @ threshold=0.5 ---
              precision    recall  f1-score   support

    licit(0)       0.98      0.99      0.98     15587
  illicit(1)       0.81      0.73      0.77      1083

    accuracy                           0.97     16670
   macro avg       0.90      0.86      0.88     16670
weighted avg       0.97      0.97      0.97     16670

